# Cropping Strategy Visualization

Compare the three cropping strategies:
1. Domain-only: Individual domains without context
2. Full module: Complete structures with pLDDT masking
3. Context-aware: Smart K=48 neighbor-preserving crops

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.cif_parser import CIFParser
from src.data.annotation_parser import AnnotationParser
from src.data.cropping.domain_only import DomainOnlyCropper, get_domain_statistics
from src.data.cropping.full_module import FullModuleCropper, compute_plddt_statistics
from src.data.cropping.context_aware import ContextAwareCropper, analyze_crop_statistics

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load annotations and sample structure
annotation_csv = Path('../fragments_for_prediction_COREONLY.csv')
parser = AnnotationParser(annotation_csv)

cif_dir = Path('../data/raw')
cif_parser = CIFParser()

# Find a sample structure
if cif_dir.exists():
    cif_files = list(cif_dir.glob('*.cif'))[:10]
    print(f'Found {len(cif_files)} sample CIF files')
else:
    print(f'CIF directory not found: {cif_dir}')
    cif_files = []

## Domain-Only Cropping

In [ ]:
if cif_files:
    domain_cropper = DomainOnlyCropper(plddt_threshold=70.0)
    
    all_domains = []
    for cif_path in cif_files:
        try:
            structure = cif_parser.parse(cif_path)
            import re
            base_id = re.sub(r'_model_\d+$', '', cif_path.stem)
            ann = parser.get(base_id)
            
            if ann:
                domains = domain_cropper.crop(structure, ann)
                all_domains.extend(domains)
        except Exception as e:
            continue
    
    if all_domains:
        stats = get_domain_statistics(all_domains)
        
        print('Domain-Only Cropping Statistics:')
        print('=' * 50)
        for domain_type, s in sorted(stats.items()):
            print(f"  {domain_type}: {s['count']} domains, "
                  f"mean length={s['mean_length']:.0f}, "
                  f"mean pLDDT={s['mean_plddt']:.1f}")

## Full Module Processing

In [ ]:
if cif_files:
    full_cropper = FullModuleCropper(
        high_confidence_threshold=70.0,
        low_confidence_threshold=50.0
    )
    
    all_modules = []
    for cif_path in cif_files:
        try:
            structure = cif_parser.parse(cif_path)
            import re
            base_id = re.sub(r'_model_\d+$', '', cif_path.stem)
            ann = parser.get(base_id)
            
            processed = full_cropper.process(structure, ann)
            if processed:
                all_modules.append(processed)
        except Exception as e:
            continue
    
    if all_modules:
        print('Full Module Statistics:')
        print('=' * 50)
        
        total_res = [m.length for m in all_modules]
        trainable_res = [m.trainable_length for m in all_modules]
        effective_res = [m.effective_length for m in all_modules]
        
        print(f'  Total residues: {np.mean(total_res):.0f} ± {np.std(total_res):.0f}')
        print(f'  Trainable (pLDDT>70): {np.mean(trainable_res):.0f} ± {np.std(trainable_res):.0f} '
              f'({np.mean(trainable_res)/np.mean(total_res)*100:.1f}%)')
        print(f'  In graph (pLDDT>50): {np.mean(effective_res):.0f} ± {np.std(effective_res):.0f} '
              f'({np.mean(effective_res)/np.mean(total_res)*100:.1f}%)')

## Context-Aware Cropping

In [ ]:
if cif_files:
    context_cropper = ContextAwareCropper(
        k_neighbors=48,
        plddt_design_threshold=70.0,
        plddt_context_threshold=50.0
    )
    
    all_crops = []
    for cif_path in cif_files:
        try:
            structure = cif_parser.parse(cif_path)
            import re
            base_id = re.sub(r'_model_\d+$', '', cif_path.stem)
            ann = parser.get(base_id)
            
            if ann:
                crop = context_cropper.crop(structure, ann)
                if crop:
                    all_crops.append(crop)
        except Exception as e:
            continue
    
    if all_crops:
        stats = analyze_crop_statistics(all_crops)
        
        print('Context-Aware Cropping Statistics:')
        print('=' * 50)
        print(f"  N crops: {stats['n_crops']}")
        print(f"  Mean total residues: {stats['mean_total_residues']:.0f}")
        print(f"  Mean design residues: {stats['mean_design_residues']:.0f}")
        print(f"  Mean context residues: {stats['mean_context_residues']:.0f}")
        print(f"  Context/design ratio: {stats['context_to_design_ratio']:.2f}")
        print(f"  Mean diversity score: {stats['mean_diversity']:.3f}")

## Comparison Visualization

In [ ]:
# Create comparison if we have data from all three strategies
if all_domains and all_modules and all_crops:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Domain-only length distribution
    ax = axes[0]
    lengths = [d.length for d in all_domains]
    ax.hist(lengths, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_xlabel('Length (residues)')
    ax.set_ylabel('Count')
    ax.set_title(f'Domain-Only\nMean: {np.mean(lengths):.0f}')
    
    # Full module trainable vs total
    ax = axes[1]
    total = [m.length for m in all_modules]
    trainable = [m.trainable_length for m in all_modules]
    ax.scatter(total, trainable, alpha=0.6)
    ax.plot([0, max(total)], [0, max(total)], 'r--', label='1:1')
    ax.set_xlabel('Total Residues')
    ax.set_ylabel('Trainable Residues (pLDDT>70)')
    ax.set_title('Full Module\nTrainable vs Total')
    ax.legend()
    
    # Context-aware design vs context
    ax = axes[2]
    design = [c.design_length for c in all_crops]
    context = [c.context_length for c in all_crops]
    ax.scatter(design, context, alpha=0.6, color='green')
    ax.set_xlabel('Design Residues')
    ax.set_ylabel('Context Residues')
    ax.set_title(f'Context-Aware\nRatio: {np.mean(context)/np.mean(design):.2f}')
    
    plt.tight_layout()
    plt.show()